# Resynex DTG — PPMI objective-endpoint model, 5-seed ensemble

Trains the headline Parkinson's model from scratch: HybridDTG/ObjectiveDTG,
targeting 3 CSF biomarkers + 4 DaTscan striatal binding ratios (7 outcomes),
conditioned on the full 19-feature aggregated panel plus 9 static covariates.

This notebook trains **5 independent seeds** (123, 42, 456, 789, 1000) and
evaluates them as an ensemble: point predictions and the survival distribution
are averaged across members, and generated samples are a proper mixture across
them (each draw picks one member at random) rather than an average of samples.

**Data you need to attach as a Kaggle Dataset before running this** (PPMI is
restricted-access and cannot be redistributed — export it yourself from
https://www.ppmi-info.org/ under your own Data Use Agreement, then upload the
five CSVs below as a private Kaggle Dataset and attach it to this notebook):

```
PPMI_Curated_Data_Cut_Public.csv
MDS-UPDRS_Part_I_05May2026.csv
MDS-UPDRS_Part_I_Patient_Questionnaire_05May2026.csv
MDS_UPDRS_Part_II__Patient_Questionnaire_05May2026.csv
MDS-UPDRS_Part_III_05May2026.csv
```

Set `PPMI_DATASET_SLUG` below to whatever you name that Kaggle Dataset — it
will be mounted read-only under `/kaggle/input/<slug>/`.

## 1. Setup — clone the training repo and install dependencies

In [ ]:
PPMI_DATASET_SLUG = "ppmi-raw-export"  # <-- change to your Kaggle Dataset's slug

!git clone --depth 1 https://github.com/SeifIslemBenrabah/Resynex-DTG-Model.git
%cd Resynex-DTG-Model
!pip install -q -r requirements.txt

## 2. Point the training code at the data

`src/preprocess_v2.py` reads `PPMI_DATA_DIR` (defaults to `data/ppmi/` if unset).
We point it at the attached Kaggle Dataset instead of copying anything.

In [ ]:
import os
os.environ["PPMI_DATA_DIR"] = f"/kaggle/input/{PPMI_DATASET_SLUG}"
assert os.path.isdir(os.environ["PPMI_DATA_DIR"]), (
    "Dataset not found — attach it via the notebook's Add Data panel "
    "and check PPMI_DATASET_SLUG matches its slug."
)
os.listdir(os.environ["PPMI_DATA_DIR"])

## 3. Load the raw cohort and visualize it

Senior-ML-engineer habit before any training run: look at the data first.
We check (a) how much of the 6-visit panel is actually observed and how that
decays over time — the missingness this model is built to handle — and
(b) the marginal distribution of every outcome the model has to reproduce,
since a badly skewed target is the first thing that breaks R² and calibration.

In [ ]:
import sys; sys.path.insert(0, "src")
import numpy as np
import matplotlib.pyplot as plt

from preprocess_v2 import load_ppmi_v2, TIMEPOINTS, FEAT_COLS

X, M, C, ET0, EI0, patients, scaler, meta = load_ppmi_v2()
print(f"patients: {X.shape[0]}   visits: {X.shape[1]}   raw features: {X.shape[2]}")

In [ ]:
# Observation rate per visit (missingness is informative here, not imputed away)
obs_rate = M.mean(axis=(0, 2))
plt.figure(figsize=(6, 3.5))
plt.plot(TIMEPOINTS, obs_rate, marker="o")
plt.xlabel("visit month"); plt.ylabel("fraction of cells observed")
plt.title("Observation rate by visit—falls off as the study progresses")
plt.ylim(0, 1); plt.grid(alpha=.3); plt.show()

In [ ]:
# Baseline distribution of the three CSF biomarkers and the four DaTscan ratios
# (raw units, undoing the StandardScaler fit inside load_ppmi_v2)
raw0 = X[:, 0, :] * scaler.scale_ + scaler.mean_
obs0 = M[:, 0, :].astype(bool)

targets = ["asyn", "tau", "ptau", "MIA_CAUDATE_L", "MIA_CAUDATE_R", "MIA_PUTAMEN_L", "MIA_PUTAMEN_R"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6)); axes = axes.ravel()
for ax, col in zip(axes, targets):
    j = FEAT_COLS.index(col)
    vals = raw0[obs0[:, j], j]
    ax.hist(vals, bins=30, color="#4c72b0")
    ax.set_title(f"{col}  (n={obs0[:, j].sum()})")
axes[-1].axis("off")
plt.suptitle("Baseline (month 0) distribution of the 7 objective-endpoint targets")
plt.tight_layout(); plt.show()

In [ ]:
# Event (motor-milestone) survival curve for the clinical event driving the
# concordance index, using the same 25-point MDS-UPDRS-III threshold as training
from lifelines import KaplanMeierFitter
kmf = KaplanMeierFitter()
kmf.fit(ET0, event_observed=EI0)
ax = kmf.plot_survival_function(figsize=(6, 4))
ax.set_xlabel("normalised time to event"); ax.set_ylabel("event-free probability")
ax.set_title(f"Motor-milestone event: {int(EI0.sum())}/{len(EI0)} patients observed to reach it")
plt.grid(alpha=.3); plt.show()

## 4. Build the aggregated 19-feature panel and inspect the training target

`src/aggregate_panel.py` turns the 59 individual MDS-UPDRS items into 9 clinical
subscale sums (kept as model input, not as a prediction target) and carries the
3 cognitive + 3 CSF + 4 DaTscan continuous features through unchanged. Only the
7 CSF/DaTscan columns (`OBJECTIVE_COLS`) are the actual prediction target.

In [ ]:
from aggregate_panel import build_aggregated, AGG_COLS, OBJECTIVE_COLS, d_agg

Xa, Ma, C_agg, agg_scaler, X69, M69, scaler69, pats = build_aggregated()
print(f"\naggregated panel d_agg={d_agg}, target columns: {OBJECTIVE_COLS}")

In [ ]:
# Correlation among the 7 targets at baseline — useful to sanity-check the
# panel before spending GPU time training on it
import pandas as pd
tgt_idx = [AGG_COLS.index(c) for c in OBJECTIVE_COLS]
df0 = pd.DataFrame(Xa[:, 0, tgt_idx], columns=OBJECTIVE_COLS)
df0[Ma[:, 0, tgt_idx].astype(bool).sum(axis=1) < len(tgt_idx)] = np.nan  # keep fully-observed rows only
corr = df0.corr()
plt.figure(figsize=(5.5, 4.5))
plt.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.xticks(range(len(OBJECTIVE_COLS)), OBJECTIVE_COLS, rotation=45, ha="right")
plt.yticks(range(len(OBJECTIVE_COLS)), OBJECTIVE_COLS)
plt.colorbar(label="Pearson r"); plt.title("Baseline correlation among the 7 targets")
plt.tight_layout(); plt.show()

## 5. Train the 5-seed ensemble

Each run is a separate, uninterrupted training from random initialization —
same architecture, same hyperparameters, only the seed changes. Roughly
10 minutes per seed on a Kaggle GPU (P100/T4), five runs ≈ under an hour total.
Configuration matches the thesis's reported run exactly: 70 epochs, 10-epoch
survival-loss warmup, `lam_var=1.0`, `lam_cd=0.5`, 6 discrete-time survival
bins (one per registry visit), event threshold of 25 summed Part III points.

In [ ]:
import subprocess, sys

SEEDS = [123, 42, 456, 789, 1000]
CKPT_DIRS = [f"outputs_final_bins6_seed{s}" for s in SEEDS]

for s, out_dir in zip(SEEDS, CKPT_DIRS):
    print(f"\n===== training seed={s} =====")
    cmd = [sys.executable, "src/train_objective.py",
           "--out", out_dir, "--epochs", "70", "--warmup", "10",
           "--batch", "16", "--lr", "3e-4", "--lam_var", "1.0",
           "--lam_cd", "0.5", "--bins", "6", "--event_thresh", "25.0",
           "--eval_every", "5", "--seed", str(s)]
    subprocess.run(cmd, check=True)

## 6. Evaluate the ensemble

Point predictions and the survival PMF are averaged across the 5 members;
generated samples are mixture-sampled (each of the `n_samples` draws picks one
member at random). PIT/coverage/KS are computed over 200 mixture draws per
cell, and MMD² is averaged over 20 independent draws/splits rather than a
single noisy point estimate, since these panels are only a few hundred rows.

In [ ]:
ckpts = ",".join(f"{d}/model.pt" for d in CKPT_DIRS)
cmd = [sys.executable, "src/metrics_objective_calibration_ensemble.py",
       "--ckpts", ckpts, "--nh", "64", "--bins", "6",
       "--out", "metrics_ppmi_ensemble.json"]
subprocess.run(cmd, check=True)

In [ ]:
import json
with open("metrics_ppmi_ensemble.json") as f:
    results = json.load(f)
print(json.dumps(results, indent=2))

## 7. Save the ensemble checkpoints and metrics

Kaggle persists everything under `/kaggle/working/` as the notebook's output —
download the 5 `model.pt` files and `metrics_ppmi_ensemble.json` from there,
or attach `/kaggle/working` as a new Dataset to reuse them in another notebook.

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/ppmi_ensemble", exist_ok=True)
for d in CKPT_DIRS:
    shutil.copy(f"{d}/model.pt", f"/kaggle/working/ppmi_ensemble/{d}.pt")
shutil.copy("metrics_ppmi_ensemble.json", "/kaggle/working/ppmi_ensemble/metrics_ppmi_ensemble.json")
print(os.listdir("/kaggle/working/ppmi_ensemble"))